# 05 -- Unweighted CNN Robustness Check

Everything identical to `01_main_experiment.ipynb`: same 21bp
position-based split (`data/splits/window_21/{hotspot,rare}/*.csv`), same
`SimpleCNN` architecture, same optimizer (Adam, lr=1e-4), same batch size
(128), same early stopping (patience=15 on val accuracy), same gradient
clipping / FP16 / seed. **The only change: inverse-frequency class weights
are removed from the loss** (`class_weights=None` in `src/train.py`'s
already-supported default, not a new code path).

**Why this experiment matters:** notebooks 03 and 04 both surfaced the same
suspicious pattern -- the weighted CNN's C>T recall in the hotspot dataset
was essentially zero (0.18%) despite C>T being the majority class, and this
was flagged as a likely confound behind (a) the counter-intuitive positive
entropy-vs-accuracy correlation in notebook 03, and (b) the CpG-rare subset
scoring 0% accuracy despite an 88.9% majority baseline in notebook 04. This
notebook tests that directly: does removing the class weighting restore
C>T recall, and how does that trade off against the other five classes and
overall accuracy?

All reused, not reimplemented: `src/data.py` (`load_split`, `MutationDataset`,
`CLASSES`, `encode_labels`), `src/model.py` (`SimpleCNN`), `src/train.py`
(`train_model`, `predict`), `src/metrics.py` (`compute_all_metrics`,
`plot_confusion_matrix`, `majority_baseline`, `mcnemar_test`).


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASSES, load_split, load_position_table, encode_labels, MutationDataset
from src.model import SimpleCNN, count_trainable_params
from src.train import train_model, predict
from src.metrics import (
    compute_all_metrics, plot_confusion_matrix, majority_baseline, mcnemar_test,
    cluster_paired_bootstrap_test, cluster_wilcoxon_paired_test,
)

WINDOW_SIZE = 21
DATASETS = ['hotspot', 'rare']

SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits', f'window_{WINDOW_SIZE}')
POSITION_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'position_table.csv')
MAIN_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'main')          # weighted CNN, for comparison
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'unweighted')

for name in DATASETS:
    os.makedirs(os.path.join(RESULTS_DIR, name), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Project root:   {PROJECT_ROOT}")
print(f"Splits dir:     {SPLITS_DIR}")
print(f"Weighted (main) results dir: {MAIN_RESULTS_DIR}")
print(f"Unweighted results dir:      {RESULTS_DIR}")
print(f"Device:         {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == 'cuda' else ""))
print(f"Classes:        {CLASSES}")


Project root:   C:\Users\danya\Documents\projects\tp53_mutation_subtype
Splits dir:     C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\window_21
Weighted (main) results dir: C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main
Unweighted results dir:      C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted
Device:         cuda (NVIDIA GeForce RTX 5090)
Classes:        ['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']


## Load and verify inputs (same 21bp splits notebook 01 used)


In [3]:
splits = {}
for name in DATASETS:
    splits[name] = {
        split: load_split(os.path.join(SPLITS_DIR, name, f'{split}.csv'))
        for split in ('train', 'val', 'test')
    }
    for split, df in splits[name].items():
        print(f"{name:>7}/{split:<5}: {len(df):>7,} rows, "
              f"{df['position_id'].nunique():>5,} positions, seq_len={df['Sequence'].str.len().iloc[0]}")

    # Unweighted training relies on this raw class distribution being visible,
    # since nothing will correct for it in the loss this time.
    train_dist = splits[name]['train']['MutationType'].value_counts(normalize=True).reindex(CLASSES)
    print(f"  train class distribution: {dict(train_dist.round(3))}\n")


hotspot/train: 467,431 rows, 5,304 positions, seq_len=21
hotspot/val  :  97,173 rows, 1,143 positions, seq_len=21
hotspot/test : 112,511 rows, 1,136 positions, seq_len=21
  train class distribution: {'C>A': np.float64(0.188), 'C>G': np.float64(0.085), 'C>T': np.float64(0.48), 'T>A': np.float64(0.059), 'T>C': np.float64(0.145), 'T>G': np.float64(0.042)}

   rare/train:  11,274 rows, 4,041 positions, seq_len=21
   rare/val  :   2,677 rows,   915 positions, seq_len=21
   rare/test :   2,362 rows,   884 positions, seq_len=21
  train class distribution: {'C>A': np.float64(0.136), 'C>G': np.float64(0.127), 'C>T': np.float64(0.36), 'T>A': np.float64(0.101), 'T>C': np.float64(0.194), 'T>G': np.float64(0.083)}



## Model architecture (unchanged, reused from `src/model.py`)


In [4]:
_sanity_model = SimpleCNN()
n_params = count_trainable_params(_sanity_model)
print(f"Trainable parameters: {n_params:,}")
assert n_params == 7206, f"Expected 7,206 trainable parameters, got {n_params:,}"
del _sanity_model


Trainable parameters: 7,206


## Train: hotspot and rare, from scratch, WITHOUT class weighting

Identical config to notebook 01 (Adam lr=1e-4, batch_size=128,
max_epochs=50, early stopping patience=15 on val accuracy,
ReduceLROnPlateau factor=0.5 patience=5, gradient clipping max_norm=1.0,
FP16 mixed precision) -- the single difference is `class_weights=None`
passed to `train_model`, so the loss is plain (unweighted) cross-entropy.


In [5]:
models = {}
histories = {}

for name in DATASETS:
    print("=" * 70)
    print(f"TRAINING (UNWEIGHTED): {name.upper()}")
    print("=" * 70)

    torch.manual_seed(SEED)  # re-seed so each model is trained independently and reproducibly

    train_ds = MutationDataset(splits[name]['train'])
    val_ds = MutationDataset(splits[name]['val'])

    model = SimpleCNN()
    model, history = train_model(
        model, train_ds, val_ds, device,
        max_epochs=50, batch_size=128, lr=1e-4,
        patience=15, lr_patience=5, lr_factor=0.5,
        grad_clip_norm=1.0, class_weights=None,   # <-- the only change from notebook 01
        seed=SEED, verbose=True,
    )

    checkpoint_path = os.path.join(RESULTS_DIR, name, 'checkpoint.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'seed': SEED,
        'dataset': name,
        'window_size': WINDOW_SIZE,
        'class_weighted': False,
        'best_val_accuracy': history['best_val_accuracy'],
        'best_epoch': history['best_epoch'],
        'stopped_epoch': history['stopped_epoch'],
        'classes': CLASSES,
    }, checkpoint_path)

    print(f"\nBest val_accuracy: {history['best_val_accuracy']:.4f} "
          f"(epoch {history['best_epoch']}, stopped at {history['stopped_epoch']})")
    print(f"Checkpoint saved -> {checkpoint_path}\n")

    models[name] = model
    histories[name] = history


TRAINING (UNWEIGHTED): HOTSPOT


epoch   1/50  train_loss=0.9106  val_loss=1.2959  val_acc=0.6012  lr=1.00e-04 *


epoch   2/50  train_loss=0.7313  val_loss=1.3688  val_acc=0.5873  lr=1.00e-04


epoch   3/50  train_loss=0.6843  val_loss=1.3912  val_acc=0.5830  lr=1.00e-04


epoch   4/50  train_loss=0.6592  val_loss=1.4664  val_acc=0.5992  lr=1.00e-04


...

epoch  13/50  train_loss=1.2018  val_loss=1.5149  val_acc=0.4419  lr=1.00e-04
epoch  14/50  train_loss=1.1694  val_loss=1.5176  val_acc=0.4419  lr=1.00e-04


epoch  15/50  train_loss=1.1487  val_loss=1.5194  val_acc=0.4419  lr=5.00e-05
epoch  16/50  train_loss=1.1536  val_loss=1.5219  val_acc=0.4359  lr=5.00e-05


epoch  17/50  train_loss=1.1376  val_loss=1.5242  val_acc=0.4023  lr=5.00e-05
epoch  18/50  train_loss=1.1355  val_loss=1.5258  val_acc=0.4419  lr=5.00e-05


epoch  19/50  train_loss=1.1251  val_loss=1.5289  val_acc=0.4359  lr=5.00e-05
epoch  20/50  train_loss=1.1175  val_loss=1.5316  val_acc=0.4023  lr=5.00e-05


epoch  21/50  train_loss=1.1161  val_loss=1.5322  val_acc=0.4643  lr=2.50e-05
epoch  22/50  train_loss=1.1067  val_loss=1.5338  val_acc=0.4643  lr=2.50e-05
Early stopping at epoch 22 (no val_accuracy improvement for 15 epochs)

Best val_accuracy: 0.4725 (epoch 7, stopped at 22)
Checkpoint saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\rare\checkpoint.pt



## Evaluate: same metrics as notebook 01

Accuracy (+95% bootstrap CI), balanced accuracy, macro F1, weighted F1, MCC,
confusion matrix -- via the same `compute_all_metrics` used in notebook 01.
Majority/random baselines are pulled in from `results/main/summary.csv`
rather than recomputed, since they depend only on the (unchanged) label
distribution, not on model weighting.


In [6]:
main_summary = pd.read_csv(os.path.join(MAIN_RESULTS_DIR, 'summary.csv')).set_index('dataset')
position_to_cluster = load_position_table(POSITION_TABLE_PATH).set_index('position_id')['cluster_id']

unweighted_metrics = {}
unweighted_pred_frames = {}

for name in DATASETS:
    test_df = splits[name]['test']
    test_ds = MutationDataset(test_df)
    y_test = test_ds.y.numpy()

    cnn_preds, cnn_probs = predict(models[name], test_ds, device)

    cluster_ids_test = position_to_cluster.loc[test_df['position_id']].values
    metrics = compute_all_metrics(y_test, cnn_preds, cluster_ids=cluster_ids_test)
    metrics['baselines'] = {
        'majority_accuracy': float(main_summary.loc[name, 'majority_accuracy']),
        'random_accuracy': float(main_summary.loc[name, 'random_accuracy']),
    }
    metrics['n_test_instances'] = int(len(y_test))
    metrics['n_test_positions'] = int(test_df['position_id'].nunique())
    metrics['window_size'] = WINDOW_SIZE
    metrics['class_weighted'] = False
    metrics['best_epoch'] = histories[name]['best_epoch']
    metrics['stopped_epoch'] = histories[name]['stopped_epoch']

    cm_path = os.path.join(RESULTS_DIR, name, 'confusion_matrix.png')
    plot_confusion_matrix(metrics['confusion_matrix'], CLASSES,
                           title=f"{name} (window={WINDOW_SIZE}bp, UNWEIGHTED) confusion matrix",
                           out_path=cm_path)

    pred_df = pd.DataFrame({
        'position_id': test_df['position_id'].values,
        'window_size': WINDOW_SIZE,
        'dataset': name,
        'true_label': [CLASSES[i] for i in y_test],
        'predicted_label': [CLASSES[i] for i in cnn_preds],
        'probabilities': list(cnn_probs),
    })[['position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities']]

    pred_path = os.path.join(RESULTS_DIR, name, 'predictions.parquet')
    pred_df.to_parquet(pred_path, engine='pyarrow', index=False)

    unweighted_metrics[name] = metrics
    unweighted_pred_frames[name] = pred_df

    print(f"{name}: accuracy={metrics['accuracy']:.4f}  mcc={metrics['mcc']:.4f}  "
          f"macro_f1={metrics['macro_f1']:.4f}  C>T recall={metrics['per_class']['C>T']['recall']:.4f}")
    print(f"  confusion matrix -> {cm_path}")
    print(f"  predictions      -> {pred_path}\n")


hotspot: accuracy=0.1602  mcc=-0.0553  macro_f1=0.1460  C>T recall=0.0864
  confusion matrix -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\hotspot\confusion_matrix.png
  predictions      -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\hotspot\predictions.parquet



rare: accuracy=0.3908  mcc=0.1404  macro_f1=0.1464  C>T recall=0.9291
  confusion matrix -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\rare\confusion_matrix.png
  predictions      -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\rare\predictions.parquet



## McNemar's tests

Two comparisons per dataset: unweighted CNN vs. majority baseline (same
comparison notebook 01 ran for the weighted model), and -- the one this
notebook exists to answer -- **unweighted CNN vs. weighted CNN**, using the
weighted model's own saved predictions from `results/main/{dataset}/predictions.parquet`.

Row alignment between the two prediction sets is verified explicitly (both
notebooks read the identical, unshuffled `test.csv`, so row order is
deterministic) rather than assumed.


In [7]:
mcnemar_results = {}

for name in DATASETS:
    test_df = splits[name]['test']
    y_test = encode_labels(test_df['MutationType'].values)

    unweighted_preds_labels = unweighted_pred_frames[name]['predicted_label'].values
    unweighted_preds_idx = encode_labels(unweighted_preds_labels)

    cluster_ids_test = position_to_cluster.loc[test_df['position_id']].values

    # vs majority baseline
    y_train = encode_labels(splits[name]['train']['MutationType'].values)
    maj_preds, majority_class = majority_baseline(y_train, len(y_test))
    vs_majority = mcnemar_test(y_test, unweighted_preds_idx, maj_preds)
    vs_majority_cluster = {
        'bootstrap': cluster_paired_bootstrap_test(y_test, unweighted_preds_idx, maj_preds, cluster_ids_test),
        'wilcoxon': cluster_wilcoxon_paired_test(y_test, unweighted_preds_idx, maj_preds, cluster_ids_test),
    }

    # vs the weighted CNN's own saved predictions (notebook 01)
    weighted_pred_path = os.path.join(MAIN_RESULTS_DIR, name, 'predictions.parquet')
    weighted_preds_df = pd.read_parquet(weighted_pred_path)

    # Verify row alignment before comparing -- both were built from the same
    # unshuffled test.csv, so position_id sequences must match exactly.
    assert list(weighted_preds_df['position_id']) == list(test_df['position_id']), (
        f"{name}: row order mismatch between weighted and unweighted predictions -- "
        "cannot run a paired McNemar test without matching rows."
    )
    weighted_preds_idx = encode_labels(weighted_preds_df['predicted_label'].values)
    vs_weighted = mcnemar_test(y_test, unweighted_preds_idx, weighted_preds_idx)
    vs_weighted_cluster = {
        'bootstrap': cluster_paired_bootstrap_test(y_test, unweighted_preds_idx, weighted_preds_idx, cluster_ids_test),
        'wilcoxon': cluster_wilcoxon_paired_test(y_test, unweighted_preds_idx, weighted_preds_idx, cluster_ids_test),
    }

    mcnemar_results[name] = {
        'unweighted_vs_majority': vs_majority,
        'unweighted_vs_weighted': vs_weighted,
    }
    cluster_test_results = {
        'unweighted_vs_majority': vs_majority_cluster,
        'unweighted_vs_weighted': vs_weighted_cluster,
    }

    print(f"{name}:")
    print(f"  unweighted vs majority: mcnemar(instance) p={vs_majority['pvalue']:.4g}  "
          f"cluster_bootstrap p={vs_majority_cluster['bootstrap']['pvalue']:.4g}  "
          f"cluster_wilcoxon p={vs_majority_cluster['wilcoxon']['pvalue']:.4g}")
    print(f"  unweighted vs weighted: mcnemar(instance) p={vs_weighted['pvalue']:.4g}  "
          f"cluster_bootstrap p={vs_weighted_cluster['bootstrap']['pvalue']:.4g}  "
          f"cluster_wilcoxon p={vs_weighted_cluster['wilcoxon']['pvalue']:.4g}")

    unweighted_metrics[name]['mcnemar'] = mcnemar_results[name]
    unweighted_metrics[name]['cluster_test'] = cluster_test_results


hotspot:
  unweighted vs majority: mcnemar(instance) p=0  cluster_bootstrap p=0.024  cluster_wilcoxon p=0.05855
  unweighted vs weighted: mcnemar(instance) p=4.343e-208  cluster_bootstrap p=0.204  cluster_wilcoxon p=0.09975
rare:
  unweighted vs majority: mcnemar(instance) p=6.137e-08  cluster_bootstrap p=0.284  cluster_wilcoxon p=0.3252
  unweighted vs weighted: mcnemar(instance) p=8.523e-45  cluster_bootstrap p=0  cluster_wilcoxon p=0.04333


## Save `metrics.json` per dataset

In [8]:
for name in DATASETS:
    metrics_path = os.path.join(RESULTS_DIR, name, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(unweighted_metrics[name], f, indent=2)
    print(f"{name}: metrics written -> {metrics_path}")


hotspot: metrics written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\hotspot\metrics.json
rare: metrics written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\rare\metrics.json


## Weighted vs. unweighted comparison summary

Side by side: same architecture/split/hyperparameters, only the loss
weighting differs.


In [9]:
comparison_rows = []
for name in DATASETS:
    w = unweighted_metrics[name]
    m = main_summary.loc[name]
    comparison_rows.append({
        'dataset': name,
        'weighting': 'unweighted',
        'accuracy': w['accuracy'],
        'balanced_accuracy': w['balanced_accuracy'],
        'macro_f1': w['macro_f1'],
        'weighted_f1': w['weighted_f1'],
        'mcc': w['mcc'],
        'ct_recall': w['per_class']['C>T']['recall'],
        'ct_precision': w['per_class']['C>T']['precision'],
        'majority_accuracy': w['baselines']['majority_accuracy'],
    })
    comparison_rows.append({
        'dataset': name,
        'weighting': 'weighted (notebook 01)',
        'accuracy': m['accuracy'],
        'balanced_accuracy': m['balanced_accuracy'],
        'macro_f1': m['macro_f1'],
        'weighted_f1': m['weighted_f1'],
        'mcc': m['mcc'],
        'ct_recall': json.load(open(os.path.join(MAIN_RESULTS_DIR, name, 'metrics.json')))['per_class']['C>T']['recall'],
        'ct_precision': json.load(open(os.path.join(MAIN_RESULTS_DIR, name, 'metrics.json')))['per_class']['C>T']['precision'],
        'majority_accuracy': m['majority_accuracy'],
    })

comparison_df = pd.DataFrame(comparison_rows, columns=[
    'dataset', 'weighting', 'accuracy', 'balanced_accuracy', 'macro_f1',
    'weighted_f1', 'mcc', 'ct_recall', 'ct_precision', 'majority_accuracy',
])

summary_path = os.path.join(RESULTS_DIR, 'summary.csv')
comparison_df.to_csv(summary_path, index=False)
print(f"Comparison summary written -> {summary_path}\n")
comparison_df


Comparison summary written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\summary.csv



,dataset,weighting,accuracy,balanced_accuracy,macro_f1,weighted_f1,mcc,ct_recall,ct_precision,majority_accuracy
0,hotspot,unweighted,0.160153,0.206491,0.145989,0.147627,-0.055329,0.086419,0.322521,0.649385
1,hotspot,weighted (notebook 01),0.132200,0.259782,0.155013,0.093986,0.066582,0.016506,0.499586,0.649385
2,rare,unweighted,0.390771,0.210133,0.146433,0.260793,0.140388,0.929078,0.426479,0.358171
3,rare,weighted (notebook 01),0.218036,0.198719,0.194176,0.221522,0.026101,0.276596,0.354545,0.358171


## Final printed summary

In [10]:
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

print("=" * 100)
print("FINAL RESULTS -- 05_unweighted_robustness (window=21bp)")
print("=" * 100)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 100)
print("C>T RECALL -- the specific quantity this experiment was designed to check")
print("=" * 100)
for name in DATASETS:
    weighted_ct_recall = json.load(open(os.path.join(MAIN_RESULTS_DIR, name, 'metrics.json')))['per_class']['C>T']['recall']
    unweighted_ct_recall = unweighted_metrics[name]['per_class']['C>T']['recall']
    print(f"{name}: weighted C>T recall={weighted_ct_recall:.4f}   "
          f"unweighted C>T recall={unweighted_ct_recall:.4f}   "
          f"delta={unweighted_ct_recall - weighted_ct_recall:+.4f}")

print("\n" + "=" * 100)
print("McNemar's tests")
print("=" * 100)
for name in DATASETS:
    res = mcnemar_results[name]
    print(f"\n{name}:")
    print(f"  unweighted CNN vs majority baseline: statistic={res['unweighted_vs_majority']['statistic']:.2f}  "
          f"p={res['unweighted_vs_majority']['pvalue']:.4g}")
    print(f"  unweighted CNN vs weighted CNN:       statistic={res['unweighted_vs_weighted']['statistic']:.2f}  "
          f"p={res['unweighted_vs_weighted']['pvalue']:.4g}")

print("\nOutputs:")
print(f"  {os.path.join(RESULTS_DIR, 'summary.csv')}")
for name in DATASETS:
    print(f"  {os.path.join(RESULTS_DIR, name, 'metrics.json')}")
    print(f"  {os.path.join(RESULTS_DIR, name, 'confusion_matrix.png')}")
    print(f"  {os.path.join(RESULTS_DIR, name, 'predictions.parquet')}")
    print(f"  {os.path.join(RESULTS_DIR, name, 'checkpoint.pt')}")


FINAL RESULTS -- 05_unweighted_robustness (window=21bp)
dataset              weighting  accuracy  balanced_accuracy  macro_f1  weighted_f1       mcc  ct_recall  ct_precision  majority_accuracy
hotspot             unweighted  0.160153           0.206491  0.145989     0.147627 -0.055329   0.086419      0.322521           0.649385
hotspot weighted (notebook 01)  0.132200           0.259782  0.155013     0.093986  0.066582   0.016506      0.499586           0.649385
   rare             unweighted  0.390771           0.210133  0.146433     0.260793  0.140388   0.929078      0.426479           0.358171
   rare weighted (notebook 01)  0.218036           0.198719  0.194176     0.221522  0.026101   0.276596      0.354545           0.358171

C>T RECALL -- the specific quantity this experiment was designed to check
hotspot: weighted C>T recall=0.0165   unweighted C>T recall=0.0864   delta=+0.0699
rare: weighted C>T recall=0.2766   unweighted C>T recall=0.9291   delta=+0.6525

McNemar's tests

hot